In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")
m1 = pd.read_sql(
    "SELECT ts_utc, open, high, low, close FROM fx_m1 "
    "WHERE pair = 'EURUSD' AND NOT bad_period ORDER BY ts_utc",
    engine, index_col="ts_utc",
)
m1.index = pd.to_datetime(m1.index, utc=True)
print(len(m1), m1.index.min(), m1.index.max())

3932734 2015-01-01 18:00:00+00:00 2025-12-31 21:57:00+00:00


In [2]:
def make_bars(m1, rule, minutes):
    bars = m1.resample(rule, label="left", closed="left").agg(
        {"open": "first", "high": "max", "low": "min", "close": "last"}
    )
    bars["n_min"] = m1["close"].resample(rule, label="left", closed="left").count()
    bars = bars[bars["n_min"] >= 0.5 * minutes].copy()   # at least half the minutes present
    return bars

b15 = make_bars(m1, "15min", 15)
b60 = make_bars(m1, "1h", 60)
print(len(b15), "15m bars |", len(b60), "1h bars")

263020 15m bars | 65773 1h bars


In [3]:
def session(hour):
    if 0 <= hour < 7:   return "asia"
    if 7 <= hour < 12:  return "london"
    if 12 <= hour < 16: return "overlap"    # London + New York both open
    if 16 <= hour < 21: return "newyork"
    return "late"                           # 21:00-24:00 UTC, thin and wide spreads

def add_features(b):
    prev_close = b["close"].shift(1)
    tr = pd.concat([
        b["high"] - b["low"],
        (b["high"] - prev_close).abs(),
        (b["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    b["atr_14"] = tr.rolling(14).mean()
    b["session"] = [session(h) for h in b.index.hour]
    return b

b15 = add_features(b15)
b60 = add_features(b60)
print(b60.groupby("session").size())
b60[["close", "atr_14"]].describe()

session
asia       19221
late        8204
london     13704
newyork    13690
overlap    10954
dtype: int64


,close,atr_14
count,65773.000000,65760.000000
mean,1.121159,0.001504
std,0.052563,0.000693
min,0.953900,0.000320
25%,1.085170,0.001032
50%,1.119380,0.001366
75%,1.162150,0.001804
max,1.254890,0.010151


In [4]:
out = []
for tf, b in [("15m", b15), ("1h", b60)]:
    t = b.reset_index().rename(columns={"index": "ts_utc"})
    t.insert(0, "timeframe", tf)
    t.insert(0, "pair", "EURUSD")
    out.append(t)
bars = pd.concat(out, ignore_index=True)

# Replace only the EURUSD rows, so GBPUSD bars (built in 08) are kept
from sqlalchemy import inspect
if inspect(engine).has_table("fx_bars"):
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM fx_bars WHERE pair = 'EURUSD'"))
    bars.to_sql("fx_bars", engine, if_exists="append", index=False, chunksize=10_000, method="multi")
else:
    bars.to_sql("fx_bars", engine, index=False, chunksize=10_000, method="multi")
    with engine.begin() as conn:
        conn.execute(text("ALTER TABLE fx_bars ADD PRIMARY KEY (pair, timeframe, ts_utc)"))

pd.read_sql("SELECT timeframe, COUNT(*) FROM fx_bars WHERE pair = 'EURUSD' GROUP BY 1", engine)

,timeframe,count
0,15m,263020
1,1h,65773
